In [0]:
from olist_lakehouse.io import read_managed_table, write_managed_table
from pyspark.sql import Window
from pyspark.sql.functions import (
    avg,
    col,
    countDistinct,
    rank,
    when,
)
from pyspark.sql.functions import (
    max as _max,
)
from pyspark.sql.functions import (
    round as _round,
)
from pyspark.sql.functions import (
    sum as _sum,
)

In [0]:
dbutils.widgets.text("catalog", "")

In [0]:
catalog = dbutils.widgets.get("catalog")

## Delivery performance table
Which states have the worst delivery performance?
Is it correlated to the seller state and delivery delay?

**Table requirements:**

- Grouping by the seller state

| Metric            | Description                                           |
| --------------------| ----------------------------------------------------|
| `seller_state`     | Seller state                                         |
| `total_orders`     | Total delivered orders                               |
| `on_time_orders`   | Orders delivered on time or before                   |
| `late_orders`      | Late delivered orders                                |
| `on_time_rate`     | `on_time_orders` / `total_orders` (`0.00` to `1.00`) |
| `avg_delay_days`   | Average `delivery_delay_days` (including negatives)  |
| `max_delay_days`   | Maximum days of delay                                |
| `performance_rank` | Ranking by `on_time_rate` descending                 |

In [0]:
orders = read_managed_table(catalog, "silver", "orders")
order_items = read_managed_table(catalog, "silver", "order_items")

In [0]:
pre_group_orders = (
    order_items.join(orders, on="order_id", how="inner")
    .select("order_id", "seller_state", "delivery_delay_days", "is_late")
    .distinct()
)

In [0]:
delivery_performance = (
    pre_group_orders.groupby("seller_state")
    .agg(
        countDistinct(col("order_id")).alias("total_orders"),
        _sum(when(col("is_late"), 0).otherwise(1)).alias("on_time_orders"),
        _sum(when(col("is_late"), 1).otherwise(0)).alias("late_orders"),
        avg(col("delivery_delay_days")).alias("avg_delay_days"),
        _max(col("delivery_delay_days")).alias("max_delay_days"),
    )
    .withColumn("on_time_rate", _round(col("on_time_orders") / col("total_orders"), 2))
)

In [0]:
wn_performance = Window.orderBy(col("on_time_rate").desc())
delivery_performance_ranked = delivery_performance.withColumn("performance_rank", rank().over(wn_performance))

In [0]:
write_managed_table(
    delivery_performance_ranked,
    mode="overwrite",
    table_full_path=f"{catalog}.gold.delivery_performance",
    options={"overwriteSchema": "true"}
)